In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d sohansakib75/cotton-4-class

In [ ]:
!unzip cotton-4-class.zip -d /content/cotton-4-class

In [ ]:
!ls /content/cotton-4-class

In [ ]:
!pip install timm torch torchvision sklearn matplotlib

In [ ]:
import os

path = "/content/cotton-4-class"  # আপনার আনজিপ করা পাথ
print("Path to dataset files:", path)

In [ ]:
import os

print("Dataset root path:", path)

print("Contents inside dataset folder:")
for item in os.listdir(path):
    item_path = os.path.join(path, item)
    if os.path.isdir(item_path):
        print(f"📂 {item}/")
    else:
        print(f"📄 {item}")

In [ ]:
import os
import glob

dataset_path = os.path.join(path, "Cotton leaf")

print("Path to Dataset folder:", dataset_path)

for subdir in sorted(os.listdir(dataset_path)):
    subpath = os.path.join(dataset_path, subdir)
    if os.path.isdir(subpath):
        image_files = glob.glob(os.path.join(subpath, "*.jpg")) + \
                      glob.glob(os.path.join(subpath, "*.jpeg")) + \
                      glob.glob(os.path.join(subpath, "*.png"))
        print(f"{subdir}: {len(image_files)} images")

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import random

dataset_path = "/content/cotton-4-class/Cotton leaf"

# Classes
classes = sorted(os.listdir(dataset_path))

# Plot 5 images per class
fig, axes = plt.subplots(len(classes), 5, figsize=(15, 12))

for i, cls in enumerate(classes):
    cls_path = os.path.join(dataset_path, cls)
    image_files = glob.glob(os.path.join(cls_path, "*.jpg")) + \
                  glob.glob(os.path.join(cls_path, "*.jpeg")) + \
                  glob.glob(os.path.join(cls_path, "*.png"))

    # Randomly pick 5 images
    sample_files = random.sample(image_files, 5)

    for j, img_path in enumerate(sample_files):
        img = plt.imread(img_path)
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 2:  # center column
            axes[i, j].set_title(cls, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

# Paths
input_dir = "/content/cotton-4-class/Cotton leaf"
output_dir = "/content/working/preprocessed_dataset"
os.makedirs(output_dir, exist_ok=True)

img_size = (224, 224)
num_samples = 5

classes = sorted(os.listdir(input_dir))
fig, axes = plt.subplots(len(classes), num_samples, figsize=(15, 3*len(classes)))

for i, cls in enumerate(classes):
    cls_path = os.path.join(input_dir, cls)
    if os.path.isdir(cls_path):
        save_cls_path = os.path.join(output_dir, cls)
        os.makedirs(save_cls_path, exist_ok=True)

        files = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        sample_files = random.sample(files, min(num_samples, len(files)))

        for j, file in enumerate(files):
            img_path = os.path.join(cls_path, file)
            img = cv2.imread(img_path)

            # 1️⃣ Resize
            img_resized = cv2.resize(img, img_size)

            # 2️⃣ Convert to YUV and apply standard histogram equalization on Y channel
            yuv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2YUV)
            yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
            img_eq = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)

            # 3️⃣ Gaussian blur to reduce noise
            img_blur = cv2.GaussianBlur(img_eq, (3, 3), 0)

            # 4️⃣ Normalize to [0,1]
            img_final = img_blur.astype(np.float32) / 255.0

            # Save preprocessed image
            save_img = (img_final * 255).astype(np.uint8)
            cv2.imwrite(os.path.join(save_cls_path, file), save_img)

            # Plot sample images
            if file in sample_files:
                axes[i, sample_files.index(file)].imshow(cv2.cvtColor(save_img, cv2.COLOR_BGR2RGB))
                axes[i, sample_files.index(file)].axis("off")
                if sample_files.index(file) == num_samples // 2:
                    axes[i, sample_files.index(file)].set_title(cls, fontsize=12)

plt.tight_layout()
plt.show()
print("\n✅ Preprocessed images saved in:", output_dir)


In [ ]:
import os
import shutil
import random

# Preprocessed dataset path
preprocessed_dir =  "/content/working/preprocessed_dataset"

# Split paths
split_base = "/content/working/cotton_split"
train_dir = os.path.join(split_base, "train")
val_dir   = os.path.join(split_base, "val")
test_dir  = os.path.join(split_base, "test")

# Create split folders
for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d, exist_ok=True)

# Split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

classes = sorted(os.listdir(preprocessed_dir))

for cls in classes:
    cls_path = os.path.join(preprocessed_dir, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(files)

    n_total = len(files)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)
    n_test = n_total - n_train - n_val

    splits = {
        train_dir: files[:n_train],
        val_dir: files[n_train:n_train+n_val],
        test_dir: files[n_train+n_val:]
    }

    for split_folder, split_files in splits.items():
        cls_split_path = os.path.join(split_folder, cls)
        os.makedirs(cls_split_path, exist_ok=True)
        for f in split_files:
            shutil.copy(os.path.join(cls_path, f), os.path.join(cls_split_path, f))

print("✅ Dataset split into 8o-10-10 and saved in:", split_base)


In [ ]:
import os

split_base = "/content/working/cotton_split"
splits = ["train", "val", "test"]

for split in splits:
    split_path = os.path.join(split_base, split)
    print(f"\n📂 {split.capitalize()} Split:")
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        num_images = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        print(f"{cls}: {num_images} images")


In [ ]:
import os
import cv2
import numpy as np
import random

train_dir = "/content/working/cotton_split/train"
aug_train_dir = "/content/working/cotton_train_aug"
os.makedirs(aug_train_dir, exist_ok=True)

# Augmentation functions
def random_flip(img):
    flip_code = random.choice([-1, 0, 1])
    return cv2.flip(img, flip_code)

def random_rotate(img):
    angle = random.uniform(-25, 25)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def random_zoom(img):
    zoom_factor = random.uniform(0.8, 1.2)
    h, w = img.shape[:2]
    new_h, new_w = int(h*zoom_factor), int(w*zoom_factor)
    img_resized = cv2.resize(img, (new_w, new_h))
    if zoom_factor < 1:
        pad_h = (h - new_h) // 2
        pad_w = (w - new_w) // 2
        img_padded = cv2.copyMakeBorder(img_resized, pad_h, h-new_h-pad_h,
                                        pad_w, w-new_w-pad_w, cv2.BORDER_REFLECT)
        return img_padded
    else:
        start_h = (new_h - h)//2
        start_w = (new_w - w)//2
        return img_resized[start_h:start_h+h, start_w:start_w+w]

def random_brightness(img):
    factor = random.uniform(0.7, 1.3)
    img = img.astype(np.float32) * factor
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

augmentations = [random_flip, random_rotate, random_zoom, random_brightness]

# Apply augmentations
classes = sorted(os.listdir(train_dir))
for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    save_cls_path = os.path.join(aug_train_dir, cls)
    os.makedirs(save_cls_path, exist_ok=True)

    for file in os.listdir(cls_path):
        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        img_path = os.path.join(cls_path, file)
        img = cv2.imread(img_path)

        # Save original
        cv2.imwrite(os.path.join(save_cls_path, file), img)

        # 3 random augmentations
        for k in range(3):
            aug_img = img.copy()
            aug_funcs = random.sample(augmentations, 4)
            for func in aug_funcs:
                aug_img = func(aug_img)
            filename, ext = os.path.splitext(file)
            aug_name = f"{filename}_aug{k+1}{ext}"
            cv2.imwrite(os.path.join(save_cls_path, aug_name), aug_img)

# Print image count per class
print("\n📊 Image count per class after augmentation:")
for cls in classes:
    cls_path = os.path.join(aug_train_dir, cls)
    count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{cls}: {count} images")

print(f"\n✅ All train images and augmented images saved in: {aug_train_dir}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, average_precision_score

# ----------------------
# Paths
# ----------------------
train_dir = "/content/working/cotton_train_aug"  # Training dataset path

# ----------------------
# Device
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Load full training dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV Parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc = [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # ----------------------
    # Model
    # ----------------------
    model = timm.create_model('mobilevit_s', pretrained=True, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=1)

    best_val_loss = float('inf')
    patience_counter = 0

    # ----------------------
    # Training
    # ----------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'swin_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print("Early stopping triggered")
                break

    # ----------------------
    # Load best model & predict on validation
    # ----------------------
    model.load_state_dict(torch.load(f'swin_fold{fold}.pth'))
    model.eval()

    def get_preds(loader):
        labels_list, preds_list, probs_list = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                labels_list.extend(labels.cpu().numpy())
                preds_list.extend(preds.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
        return np.array(labels_list), np.array(preds_list), np.array(probs_list)

    val_labels, val_preds, val_probs = get_preds(val_loader)
    val_labels_onehot = np.eye(num_classes)[val_labels]

    # ----------------------
    # Metrics
    # ----------------------
    acc = np.mean(val_labels == val_preds)
    kappa = cohen_kappa_score(val_labels, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))

    print(f"Fold {fold} | Acc: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | PR AUC: {np.mean(pr_auc_list):.4f}")

    # ----------------------
    # Clear GPU memory
    # ----------------------
    del model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# 5-Fold CV Summary
# ----------------------
print("\n===== 5-Fold CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, average_precision_score

# ----------------------
# Paths
# ----------------------
train_dir = "/content/working/cotton_train_aug"  # Training dataset path

# ----------------------
# Device
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Load full training dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV Parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc = [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # ----------------------
    # Model
    # ----------------------
    model = timm.create_model('tiny_vit_5m_224', pretrained=True, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=1)

    best_val_loss = float('inf')
    patience_counter = 0

    # ----------------------
    # Training
    # ----------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'swin_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print("Early stopping triggered")
                break

    # ----------------------
    # Load best model & predict on validation
    # ----------------------
    model.load_state_dict(torch.load(f'swin_fold{fold}.pth'))
    model.eval()

    def get_preds(loader):
        labels_list, preds_list, probs_list = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                labels_list.extend(labels.cpu().numpy())
                preds_list.extend(preds.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
        return np.array(labels_list), np.array(preds_list), np.array(probs_list)

    val_labels, val_preds, val_probs = get_preds(val_loader)
    val_labels_onehot = np.eye(num_classes)[val_labels]

    # ----------------------
    # Metrics
    # ----------------------
    acc = np.mean(val_labels == val_preds)
    kappa = cohen_kappa_score(val_labels, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))

    print(f"Fold {fold} | Acc: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | PR AUC: {np.mean(pr_auc_list):.4f}")

    # ----------------------
    # Clear GPU memory
    # ----------------------
    del model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# 5-Fold CV Summary
# ----------------------
print("\n===== 5-Fold CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, average_precision_score

# ----------------------
# Paths
# ----------------------
train_dir = "/content/working/cotton_train_aug"  # Training dataset path

# ----------------------
# Device
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Load full training dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV Parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc = [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # ----------------------
    # Model
    # ----------------------
    model = timm.create_model('levit_192', pretrained=True, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)

    best_val_loss = float('inf')
    patience_counter = 0

    # ----------------------
    # Training
    # ----------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'swin_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print("Early stopping triggered")
                break

    # ----------------------
    # Load best model & predict on validation
    # ----------------------
    model.load_state_dict(torch.load(f'swin_fold{fold}.pth'))
    model.eval()

    def get_preds(loader):
        labels_list, preds_list, probs_list = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                labels_list.extend(labels.cpu().numpy())
                preds_list.extend(preds.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
        return np.array(labels_list), np.array(preds_list), np.array(probs_list)

    val_labels, val_preds, val_probs = get_preds(val_loader)
    val_labels_onehot = np.eye(num_classes)[val_labels]

    # ----------------------
    # Metrics
    # ----------------------
    acc = np.mean(val_labels == val_preds)
    kappa = cohen_kappa_score(val_labels, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))

    print(f"Fold {fold} | Acc: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | PR AUC: {np.mean(pr_auc_list):.4f}")

    # ----------------------
    # Clear GPU memory
    # ----------------------
    del model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# 5-Fold CV Summary
# ----------------------
print("\n===== 5-Fold CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, average_precision_score

# ----------------------
# Paths
# ----------------------
train_dir = "/content/working/cotton_train_aug"  # Training dataset path

# ----------------------
# Device
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Load full training dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV Parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc = [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # ----------------------
    # Model
    # ----------------------
    model = timm.create_model('maxvit_tiny_tf_224', pretrained=True, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=1)

    best_val_loss = float('inf')
    patience_counter = 0

    # ----------------------
    # Training
    # ----------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'swin_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print("Early stopping triggered")
                break

    # ----------------------
    # Load best model & predict on validation
    # ----------------------
    model.load_state_dict(torch.load(f'swin_fold{fold}.pth'))
    model.eval()

    def get_preds(loader):
        labels_list, preds_list, probs_list = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                labels_list.extend(labels.cpu().numpy())
                preds_list.extend(preds.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
        return np.array(labels_list), np.array(preds_list), np.array(probs_list)

    val_labels, val_preds, val_probs = get_preds(val_loader)
    val_labels_onehot = np.eye(num_classes)[val_labels]

    # ----------------------
    # Metrics
    # ----------------------
    acc = np.mean(val_labels == val_preds)
    kappa = cohen_kappa_score(val_labels, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))

    print(f"Fold {fold} | Acc: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | PR AUC: {np.mean(pr_auc_list):.4f}")

    # ----------------------
    # Clear GPU memory
    # ----------------------
    del model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# 5-Fold CV Summary
# ----------------------
print("\n===== 5-Fold CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, average_precision_score

# ----------------------
# Paths
# ----------------------
train_dir = "/content/working/cotton_train_aug"  # Training dataset path

# ----------------------
# Device
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Load full training dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV Parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc = [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subset datasets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # ----------------------
    # Model
    # ----------------------
    model = timm.create_model('deit_small_patch16_224', pretrained=True, num_classes=num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)

    best_val_loss = float('inf')
    patience_counter = 0

    # ----------------------
    # Training
    # ----------------------
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'swin_fold{fold}.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print("Early stopping triggered")
                break

    # ----------------------
    # Load best model & predict on validation
    # ----------------------
    model.load_state_dict(torch.load(f'swin_fold{fold}.pth'))
    model.eval()

    def get_preds(loader):
        labels_list, preds_list, probs_list = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                labels_list.extend(labels.cpu().numpy())
                preds_list.extend(preds.cpu().numpy())
                probs_list.extend(probs.cpu().numpy())
        return np.array(labels_list), np.array(preds_list), np.array(probs_list)

    val_labels, val_preds, val_probs = get_preds(val_loader)
    val_labels_onehot = np.eye(num_classes)[val_labels]

    # ----------------------
    # Metrics
    # ----------------------
    acc = np.mean(val_labels == val_preds)
    kappa = cohen_kappa_score(val_labels, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))

    print(f"Fold {fold} | Acc: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | PR AUC: {np.mean(pr_auc_list):.4f}")

    # ----------------------
    # Clear GPU memory
    # ----------------------
    del model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# 5-Fold CV Summary
# ----------------------
print("\n===== 5-Fold CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")


In [ ]:
import os, time, gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ----------------------
# Paths & Device
# ----------------------
train_dir = "/content/working/cotton_train_aug"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------
# Transform
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------
# Dataset
# ----------------------
full_dataset = datasets.ImageFolder(train_dir, transform=transform)
full_labels = np.array([label for _, label in full_dataset])
class_names = full_dataset.classes
num_classes = len(class_names)

# ----------------------
# CV parameters
# ----------------------
num_epochs = 30
batch_size = 32
n_splits = 5
model_names = ['mobilevit_s', 'tiny_vit_5m_224']

# Metrics storage
fold_acc, fold_kappa, fold_roc_auc, fold_pr_auc, fold_specificity = [], [], [], [], []

# ----------------------
# Stratified K-Fold
# ----------------------
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(full_labels)), full_labels), 1):
    print(f"\n===== Fold {fold}/{n_splits} =====")

    # Subsets
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

    # ----------------------
    # Train all models for this fold
    # ----------------------
    trained_models = []
    for name in model_names:
        print(f"\n--- Training {name} ---")
        model = timm.create_model(name, pretrained=True, num_classes=num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)

        best_val_loss = float('inf')
        patience_counter = 0
        best_state = None

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            # Validation
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
            val_loss /= len(val_loader)
            scheduler.step(val_loss)

            print(f"Epoch [{epoch+1}/{num_epochs}] | Val Loss: {val_loss:.4f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 3:
                    break

        # Load best weights
        model.load_state_dict(best_state)
        model.eval()
        trained_models.append(model)

    # ----------------------
    # Soft Voting Ensemble on Validation Set
    # ----------------------
    def get_ensemble_preds(loader, models):
        all_probs, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                ensemble_probs = torch.zeros(inputs.size(0), num_classes).to(device)
                for m in models:
                    outputs = m(inputs)
                    probs = torch.softmax(outputs, dim=1)
                    ensemble_probs += probs
                ensemble_probs /= len(models)
                all_probs.append(ensemble_probs.cpu().numpy())
                all_labels.extend(labels.numpy())
        return np.vstack(all_probs), np.array(all_labels)

    val_probs, val_labels_array = get_ensemble_preds(val_loader, trained_models)
    val_preds = np.argmax(val_probs, axis=1)
    val_labels_onehot = np.eye(num_classes)[val_labels_array]

    # ----------------------
    # Metrics (ensemble-level)
    # ----------------------
    acc = np.mean(val_labels_array == val_preds)
    kappa = cohen_kappa_score(val_labels_array, val_preds)
    roc_auc_list = [roc_auc_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]
    pr_auc_list  = [average_precision_score(val_labels_onehot[:,i], val_probs[:,i]) for i in range(num_classes)]

    # Confusion matrix & specificity (macro-average)
    cm = confusion_matrix(val_labels_array, val_preds)
    TN = np.diag(cm)
    FP = cm.sum(axis=0) - TN
    specificity = TN / (TN + FP + 1e-8)

    fold_acc.append(acc)
    fold_kappa.append(kappa)
    fold_roc_auc.append(np.mean(roc_auc_list))
    fold_pr_auc.append(np.mean(pr_auc_list))
    fold_specificity.append(np.mean(specificity))

    print(f"\nFold {fold} Results (Ensemble):")
    print(f"Accuracy: {acc:.4f} | Kappa: {kappa:.4f} | ROC AUC: {np.mean(roc_auc_list):.4f} | "
          f"PR AUC: {np.mean(pr_auc_list):.4f} | Specificity: {np.mean(specificity):.4f}")

    # Free memory
    del trained_models, model, optimizer, scheduler, criterion, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

# ----------------------
# Cross-Validation Summary
# ----------------------
print("\n===== 5-Fold Ensemble CV Summary =====")
print(f"Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Kappa: {np.mean(fold_kappa):.4f} ± {np.std(fold_kappa):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_auc):.4f} ± {np.std(fold_roc_auc):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_auc):.4f} ± {np.std(fold_pr_auc):.4f}")
print(f"Mean Specificity: {np.mean(fold_specificity):.4f} ± {np.std(fold_specificity):.4f}")
